<a href="https://colab.research.google.com/github/dmecoyfin/dmeyf2026/blob/ensembles_2026/ensembles/z440_RandomForest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4. Ensembles de Arboles de Decision

## 4.03 Random Forest

*Random Forest* es un algoritmo de ensembles de arboles de decision creado por Leo Brieman en 1995-2006
https://link.springer.com/content/pdf/10.1023/a:1010933404324.pdf

La página original es:
https://www.stat.berkeley.edu/~breiman/RandomForests/cc_home.htm

Dos buenos videos para seguir el paso a paso de Random Forest y aplicaciones:
* https://www.youtube.com/watch?v=J4Wdy0Wc_xQ
* https://www.youtube.com/watch?v=sQ870aTKqiM

Qué tipo de perturbaciones se realizan en Random Forest

*   Se perturba el dataset, con la técnica de bagging = Bootstrap Aggregating
*   Tambien se perturba el algoritmo, utiliza random en cada split

Cada arbolito de Random Forest se entrena sobre un dataset perturbado, que tiene :
* todas las columnas originales (esta es una GRAN diferencia con  Arboles Azarosos)
* la misma *cantidad* de registros del dataset original, PERO generados por la técnica de sampleo con reposición del dataset original.

A pesar de que Leo Brieman es también el inventor de CART (Classification and Regression Trees) Random Forest no corre el algoritmo CART de la libreria rpart, sino un CART perturbado, en donde cada split NO se hace sobre todos los campos del dataset, sino sobre un csubconjunto tomado al azar, esa cantidad es el hiperparámetro *mtry*

#### 4.03.1  Seteo del ambiente en Google Colab

# URLS


In [64]:
# install.packages("here")  # solo la primera vez
library(here)

root <- here()
# dataset <- fread(file.path(path_base, "competencia_01_crudo.csv"))

In [65]:
list.files(root)

[1] "arboles"          "CazaTalentos"     "DATA"             "ensembles"       
[5] "git-zero-to-hero" "mis_pruebas"      "monday"           "zero2hero"

In [66]:
dataset_folder <- file.path(root, "DATA", "DATASETS")
exp_folder <- file.path(root, "DATA", "EXP")

dataset_url <- file.path(dataset_folder,"competencia_01_crudo.csv")

## Generacion de la clase_ternaria

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Tipe -> Runtime type -> R

In [5]:
require( "data.table" )

# leo el dataset
# dataset <- fread("/content/datasets/competencia_01_crudo.csv" )
dataset <- fread(dataset_url)


# calculo el periodo0 consecutivo
dsimple <- dataset[, list(
    "pos" = .I,
    numero_de_cliente,
    periodo0 = as.integer(foto_mes/100)*12 +  foto_mes%%100 ) ]


# ordeno
setorder( dsimple, numero_de_cliente, periodo0 )

# calculo topes
periodo_ultimo <- dsimple[, max(periodo0) ]
periodo_anteultimo <- periodo_ultimo - 1


# calculo los leads de orden 1 y 2
dsimple[, c("periodo1", "periodo2") :=
    shift(periodo0, n=1:2, fill=NA, type="lead"),  numero_de_cliente ]

# assign most common class values = "CONTINUA"
dsimple[ periodo0 < periodo_anteultimo, clase_ternaria := "CONTINUA" ]

# calculo BAJA+1
dsimple[ periodo0 < periodo_ultimo &
    ( is.na(periodo1) | periodo0 + 1 < periodo1 ),
    clase_ternaria := "BAJA+1" ]

# calculo BAJA+2
dsimple[ periodo0 < periodo_anteultimo & (periodo0+1 == periodo1 )
    & ( is.na(periodo2) | periodo0 + 2 < periodo2 ),
    clase_ternaria := "BAJA+2" ]


# pego el resultado en el dataset original y grabo
setorder( dsimple, pos )
dataset[, clase_ternaria := dsimple$clase_ternaria ]

fwrite( dataset,
    # file =  "/content/datasets/competencia_01.csv.gz",
    file = file.path(dataset_folder,"competencia_01.csv.gz"),
    sep = ","
)

In [6]:
setorder( dataset, foto_mes, clase_ternaria, numero_de_cliente)
dataset[, .N, list(foto_mes, clase_ternaria)]

foto_mes,clase_ternaria,N
<int>,<chr>,<int>
202103,BAJA+1,1019
202103,BAJA+2,960
202103,CONTINUA,160921
202104,BAJA+1,964
202104,BAJA+2,1139
202104,CONTINUA,161181
202105,BAJA+1,1143
202105,BAJA+2,870
202105,CONTINUA,161755


### 4.04  Random Forest, una corrida

El tiempo de corrida de este punto es de alrededor de 8 minutos

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [7]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Mon Sep 07 10:25:51 2026"

In [8]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,707763,37.8,1413002,75.5,1413002,75.5
Vcells,1373688,10.5,297539111,2270.1,336603967,2568.1


**ranger** es una de las muchas librerías en lenguage R que implementa el algoritmo *Random Forest*, tiene la ventaja que corre el paralelo, utilizando todos los nucleos del procesador.

In [31]:
# cargo las librerias que necesito
require("data.table")
require("rpart")

if(!require("R.utils")) install.packages("R.utils")
require("R.utils")

# ranger se usa para procesar
if( !require("ranger") ) install.packages("ranger")
require("ranger")

# randomForest  solo se usa para imputar nulos
if( !require("randomForest") ) install.packages("randomForest")
require("randomForest")

Loading required package: ranger

Warning message in library(package, lib.loc = lib.loc, character.only = TRUE, logical.return = TRUE, :
“there is no package called ‘ranger’”
Warning message in install.packages("ranger"):
“installation of package ‘ranger’ had non-zero exit status”
Updating HTML index of packages in '.Library'

Making 'packages.html' ...
 done

Loading required package: ranger

Warning message in library(package, lib.loc = lib.loc, character.only = TRUE, logical.return = TRUE, :
“there is no package called ‘ranger’”


Aqui debe cargar SU semilla primigenia y

In [85]:
PARAM <- list()
PARAM$experimento <- 440
PARAM$semilla_primigenia <- 230047

# training y future
PARAM$train <- c(202104)
PARAM$future <- c(202106)

# PARAM$ranger$num.trees <- 300 # cantidad de arboles
# PARAM$ranger$mtry <- 13 # cantidad de atributos que participan en cada split
# PARAM$ranger$min.node.size <- 50 # tamaño minimo de las hojas
# PARAM$ranger$max.depth <- 10 # 0 significa profundidad infinita

#resultados de bayesiana
PARAM$ranger$num.trees <- 485 # cantidad de arboles
PARAM$ranger$mtry <- 11 # cantidad de atributos que participan en cada split
PARAM$ranger$min.node.size <- 396 # tamaño minimo de las hojas
PARAM$ranger$max.depth <- 20 # 0 significa profundidad infinita

PARAM$semilla_kaggle <- 314159

In [86]:
# particionar agrega una columna llamada fold a un dataset
#   que consiste en una particion estratificada segun agrupa
# particionar( data=dataset, division=c(70,30),
#  agrupa=clase_ternaria, seed=semilla)   crea una particion 70, 30

particionar <- function(data, division, agrupa= "", campo= "fold", start= 1, seed= NA) {
  if (!is.na(seed)) set.seed(seed, "L'Ecuyer-CMRG")

  bloque <- unlist(mapply(
    function(x, y) {rep(y, x)},division, seq(from= start, length.out= length(division))))

  data[, (campo) := sample(rep(bloque,ceiling(.N / length(bloque))))[1:.N],by= agrupa]
}

In [87]:
# iniciliazo el dataset de realidad, para medir ganancia
realidad_inicializar <- function( pfuture, pparam) {

  # datos para verificar la ganancia
  drealidad <- pfuture[, list(numero_de_cliente, foto_mes, clase_ternaria)]

  particionar(drealidad,
    division= c(3, 7),
    agrupa= "clase_ternaria",
    seed= PARAM$semilla_kaggle
  )

  return( drealidad )
}

In [88]:
# evaluo ganancia en los datos de la realidad

realidad_evaluar <- function( prealidad, pprediccion) {

  prealidad[ pprediccion,
    on= c("numero_de_cliente", "foto_mes"),
    predicted:= i.Predicted
  ]

  tbl <- prealidad[, list("qty"=.N), list(fold, predicted, clase_ternaria)]

  res <- list()
  res$public  <- tbl[fold==1 & predicted==1L, sum(qty*ifelse(clase_ternaria=="BAJA+2", 1072500, -27500))]/0.3
  res$private <- tbl[fold==2 & predicted==1L, sum(qty*ifelse(clase_ternaria=="BAJA+2", 1072500, -27500))]/0.7
  res$total <- tbl[predicted==1L, sum(qty*ifelse(clase_ternaria=="BAJA+2", 1072500, -27500))]

  prealidad[, predicted:=NULL]
  return( res )
}

In [89]:
# carpeta de trabajo
# setwd("/content/buckets/b1/exp")
setwd(exp_folder)

experimento_folder <- paste0("KA", PARAM$experimento)
dir.create(experimento_folder, showWarnings=FALSE)
# setwd( paste0("/content/buckets/b1/exp/", experimento_folder ))
setwd(file.path(exp_folder, experimento_folder))


In [90]:
experimento_folder

[1] "KA440"

In [91]:
# lectura del dataset
# dataset <- fread("/content/datasets/competencia_01.csv.gz")
dataset <- fread(file.path(dataset_folder,"competencia_01.csv.gz"))

In [92]:
#  estas dos lineas estan relacionadas con el Data Drifting
# asigno un valor muy negativo

if( "Master_Finiciomora" %in% colnames(dataset) )
  dataset[ is.na(Master_Finiciomora) , Master_Finiciomora := -999 ]

if( "Visa_Finiciomora" %in% colnames(dataset) )
  dataset[ is.na(Visa_Finiciomora) , Visa_Finiciomora :=  -999 ]


In [93]:
# defino los dataset de entrenamiento y aplicacion
dtrain <- dataset[foto_mes %in% PARAM$train]

In [94]:
# mes donde voy a aplicar el modelo
dfuture <- dataset[foto_mes %in% PARAM$future]
setorder(dfuture, numero_de_cliente, foto_mes)

In [95]:
# inicilizo el dataset  drealidad
drealidad <- realidad_inicializar( dfuture, PARAM)

In [96]:
# quito clase ternaria de donde voy a aplicar el modelo
dfuture[, clase_ternaria:= NULL]

In [97]:
set.seed(PARAM$semilla_primigenia,"L'Ecuyer-CMRG" ) # Establezco la semilla aleatoria

# ranger necesita la clase de tipo factor
factorizado <- as.factor(dtrain$clase_ternaria)
dtrain[, clase_ternaria := factorizado]

In [98]:
# Ranger NO acepta valores nulos
# Leo Breiman, ¿por que le temias a los nulos?
# imputo los nulos, ya que ranger no acepta nulos
dtrain <- na.roughfix(dtrain)


In [99]:
setorder(dtrain, clase_ternaria) # primero quedan los BAJA+1, BAJA+2, CONTINUA

# genero el modelo de Random Forest llamando a ranger()
modelo <- ranger(
  formula= "clase_ternaria ~ .",
  data= dtrain,
  probability= TRUE, # para que devuelva las probabilidades
  num.trees= PARAM$ranger$num.trees,
  mtry= PARAM$ranger$mtry,
  min.node.size= PARAM$ranger$min.node.size,
  max.depth= PARAM$ranger$max.depth
)


Growing trees.. Progress: 35%. Estimated remaining time: 57 seconds.
Growing trees.. Progress: 70%. Estimated remaining time: 26 seconds.


In [100]:
# Carpinteria necesaria sobre  dfuture
# como quiere la Estadistica Clasica, imputar nulos por separado
# ( aunque en este caso ya tengo los datos del futuro de antemano
#  pero bueno, sigamos el librito de estos fundamentalistas a rajatabla ...

dfuture <- na.roughfix(dfuture)

In [101]:
tb_prediccion <- dfuture[, list(numero_de_cliente, foto_mes)]

In [102]:
# aplico el modelo a los datos que no tienen clase
# aplico el modelo recien creado a los datos del futuro
prediccion <- predict(modelo, dfuture)

tb_prediccion[, prob := prediccion$predictions[, "BAJA+2"] ]

In [103]:
tb_prediccion[, Predicted := as.numeric(prob > (1/40))]

In [104]:
res <- realidad_evaluar( drealidad, tb_prediccion)

In [105]:
cat( " TOTAL=", res$total,
  " Public=", res$public,
  " Private=", res$private,
  "\n",
  sep= ""
)

 TOTAL=385247500 Public=396550000 Private=380403571


In [50]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Mon Sep 07 10:44:09 2026"

Reportar el resultado  TOTAL en la hoja C4-Random Forest de la planilla colaborativa

Usted NO reportará la ganancia del Public ni tampoco la del Private, por ahora es simplemente para que perciba la variabilidad existente y comience a tener plena conciencia de los fenómenos que observó en la asignatura  Data Mining del cuatrimestre anterior.



---



### 4.05  Random Forest  optimizacion de hiperparámetros

Random Forest es un algoritmo que quedó obsoleto luego de la aparición de  XGBoost y LightGBM, debido a lo lento de las librerías que lo implementan.
<br> El siguiente script se brinda simplemente a modo pedagógico, advirtiendo a l@s alumn@s que demanda más de 24 horas para correr, y los resultados no son brillantes.

limpio el ambiente de R

In [51]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Mon Sep 07 10:45:40 2026"

In [52]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,2033961,108.7,4124080,220.3,4043859,216.0
Vcells,3765529,28.8,238031289,1816.1,336603967,2568.1


**ranger** es una de las muchas librerías en lenguage R que implementa el algoritmo *Random Forest*, tiene la ventaja que corre el paralelo, utilizando todos los nucleos del procesador.

In [70]:
# cargo las librerias que necesito
require("data.table")
require("rpart")
require("parallel")

if(!require("R.utils")) install.packages("R.utils")
require("R.utils")

if( !require("primes") ) install.packages("primes")
require("primes")

if( !require("rlist") ) install.packages("rlist")
require("rlist")

# ranger se usa para procesar
if( !require("ranger") ) install.packages("ranger")
require("ranger")

# randomForest  solo se usa para imputar nulos
if( !require("randomForest") ) install.packages("randomForest")
require("randomForest")


if( !require("DiceKriging") ) install.packages("DiceKriging")
require("DiceKriging")

if( !require("mlrMBO") ) install.packages("mlrMBO")
require("mlrMBO")


Loading required package: primes



Aqui debe cargar SU semilla primigenia y

In [56]:
PARAM <- list()
PARAM$experimento <- 450
PARAM$semilla_primigenia <- 230047

PARAM$hyperparametertuning$iteraciones <- 5
PARAM$hyperparametertuning$xval_folds <- 5
PARAM$hyperparametertuning$POS_ganancia <- 1072500
PARAM$hyperparametertuning$NEG_ganancia <- -27500

# Estructura que define los hiperparámetros y sus rangos
#  la letra L al final significa ENTERO
# max.depth 0 significa profundidad infinita
PARAM$hyperparametertuning$hs <- makeParamSet(
  makeIntegerParam("num.trees", lower= 20L, upper= 500L),
  makeIntegerParam("max.depth", lower= 1L, upper= 30L),
  makeIntegerParam("min.node.size", lower= 1L, upper= 1000L),
  makeIntegerParam("mtry", lower= 2L, upper= 50L)
)

# training
PARAM$train <- c(202104)

In [58]:
getwd()

[1] "/home/marco/code/DMEyF/dmeyf2026/DATA/EXP/KA440"

In [59]:
# graba a un archivo los componentes de lista
# para el primer registro, escribe antes los titulos

loguear <- function(
    reg, arch= NA, folder= "./work/",
    ext= ".txt", verbose= TRUE) {

  archivo <- arch
  if (is.na(arch)) archivo <- paste0(folder, substitute(reg), ext)

  if (!file.exists(archivo)) # Escribo los titulos
    {
      linea <- paste0(
        "fecha\t",
        paste(list.names(reg), collapse= "\t"), "\n"
      )

      cat(linea, file= archivo)
    }

  linea <- paste0(
    format(Sys.time(), "%Y%m%d %H%M%S"), "\t", # la fecha y hora
    gsub(", ", "\t", toString(reg)), "\n"
  )

  cat(linea, file= archivo, append= TRUE) # grabo al archivo

  if (verbose) cat(linea) # imprimo por pantalla
}


In [60]:
# particionar agrega una columna llamada fold a un dataset
#  que consiste en una particion estratificada segun agrupa
# particionar( data=dataset, division=c(70,30),
#  agrupa=clase_ternaria, seed=semilla)   crea una particion 70, 30
# particionar( data=dataset, division=c(1,1,1,1,1),
#   agrupa=clase_ternaria, seed=semilla)   divide el dataset en 5 particiones

particionar <- function(
    data, division, agrupa= "",
    campo= "fold", start= 1, seed= NA) {

  if (!is.na(seed)) set.seed(seed)

  bloque <- unlist(mapply(function(x, y) {
    rep(y, x)
  }, division, seq(from= start, length.out= length(division))))

  data[, (campo) := sample(rep(bloque, ceiling(.N / length(bloque))))[1:.N],
    by= agrupa
  ]
}


In [ ]:
# es un paso del Cross Validation
# utiliza el fold  fold_test para testear y el resto para entrenar
cores_disponibles <- parallel::detectCores(logical = TRUE)
threads_ranger <- max(1L, floor(cores_disponibles * 2/3))

ranger_Simple <- function(fold_test, pdata, param) {
  # genero el modelo

  set.seed(PARAM$semillas[2])

  modelo <- ranger(
    formula= "clase_binaria ~ .",
    data= pdata[fold != fold_test],
    probability= TRUE, # para que devuelva las probabilidades
    num.trees= param$num.trees,
    mtry= param$mtry,
    min.node.size= param$min.node.size,
    max.depth= param$max.depth,
    #num.threads = threads_ranger #habilitar mas cores
  )

  prediccion <- predict(modelo, pdata[fold == fold_test])

  ganancia_testing <- pdata[
    fold == fold_test,
    sum((prediccion$predictions[, "POS"] > 1 / 40) *
      ifelse(clase_binaria == "POS",
        PARAM$hyperparametertuning$POS_ganancia,
        PARAM$hyperparametertuning$NEG_ganancia
      ))
  ]

  return(ganancia_testing)
}


In [62]:
# realiza Cross Validation, promediando las ganancias de los folds de testing

ranger_CrossValidation <- function(
    data, param,
    pcampos_buenos, qfolds, pagrupa, semilla) {

  divi <- rep(1, qfolds)
  particionar(data, divi, seed= semilla, agrupa= pagrupa)

  ganancias <- mcmapply(ranger_Simple,
    seq(qfolds), # 1 2 3 4 5
    MoreArgs= list(data, param),
    SIMPLIFY= FALSE,
    mc.cores= 1
  ) # dejar esto en  1, porque ranger ya corre en paralelo

  data[, fold := NULL] # elimino el campo fold

  # devuelvo la ganancia promedio normalizada
  ganancia_promedio <- mean(unlist(ganancias))
  ganancia_promedio_normalizada <- ganancia_promedio * qfolds

  return(ganancia_promedio_normalizada)
}

In [63]:
# esta funcion solo puede recibir los parametros que se estan optimizando
# el resto de los parametros se pasan como variables globales

EstimarGanancia_ranger <- function(x) {
  GLOBAL_iteracion <<- GLOBAL_iteracion + 1

  xval_folds <- PARAM$hyperparametertuning$xval_folds

  ganancia <- ranger_CrossValidation(dataset,
    param= x,
    qfolds= xval_folds,
    pagrupa= "clase_binaria",
    semilla= PARAM$semillas[1]
  )

  # logueo
  xx <- x
  xx$xval_folds <- xval_folds
  xx$ganancia <- ganancia
  xx$iteracion <- GLOBAL_iteracion
  loguear(xx, arch= klog)

  # si es ganancia superadora la almaceno en mejor
  if( ganancia > GLOBAL_mejor ) {
    GLOBAL_mejor <<- ganancia
    loguear(xx, arch= klog_mejor)
  }


  return(ganancia)
}


aqui se inicia el programa

In [67]:
# carpeta de trabajo
# setwd("/content/buckets/b1/exp")
setwd(exp_folder)
experimento_folder <- paste0("HT", PARAM$experimento)
dir.create(experimento_folder, showWarnings=FALSE)
setwd( file.path(exp_folder, experimento_folder ))

In [71]:
# genero numeros primos
primos <- generate_primes(min= 100000, max= 1000000)
set.seed(PARAM$semilla_primigenia) # inicializo
# me quedo con PARAM$qsemillas   semillas
PARAM$semillas <- sample(primos, 2 )


In [72]:
# lectura del dataset
# dataset <- fread("/content/datasets/competencia_01.csv.gz")
dataset <- fread(file.path(dataset_folder, "competencia_01.csv.gz"))

In [73]:
# solo trabajo con  training
dataset <- dataset[foto_mes %in% PARAM$train ]

In [74]:
#  estas dos lineas estan relacionadas con el Data Drifting
# asigno un valor muy negativo

if( "Master_Finiciomora" %in% colnames(dataset) )
  dataset[ is.na(Master_Finiciomora) , Master_Finiciomora := -999 ]

if( "Visa_Finiciomora" %in% colnames(dataset) )
  dataset[ is.na(Visa_Finiciomora) , Visa_Finiciomora :=  -999 ]


In [75]:
set.seed(PARAM$semilla_primigenia,"L'Ecuyer-CMRG" ) # Establezco la semilla aleatoria

In [76]:
# en estos archivos quedan los resultados
kbayesiana <- paste0("HT", PARAM$experimento, ".RDATA")
klog <- paste0("HT", PARAM$experimento, ".txt")
klog_mejor <- paste0("HT", PARAM$experimento, "_mejor.txt")

GLOBAL_iteracion <- 0 # inicializo la variable global
GLOBAL_mejor <- -Inf

# si ya existe el archivo log, traigo hasta donde llegue
if (file.exists(klog)) {
  tabla_log <- fread(klog)
  GLOBAL_iteracion <- nrow(tabla_log)
}


In [77]:
# paso a trabajar con clase binaria POS={BAJA+2}   NEG={BAJA+1, CONTINUA}
dataset[, clase_binaria :=
  as.factor(ifelse(clase_ternaria == "BAJA+2", "POS", "NEG"))]

dataset[, clase_ternaria := NULL] # elimino la clase_ternaria, ya no la necesito


In [78]:
# Ranger NO acepta valores nulos
# Leo Breiman, ¿por que le temias a los nulos?
# imputo los nulos, ya que ranger no acepta nulos

dataset <- na.roughfix(dataset)

In [79]:
# Aqui comienza la configuracion de la Bayesian Optimization

configureMlr(show.learner.output = FALSE)

funcion_optimizar <- EstimarGanancia_ranger

# configuro la busqueda bayesiana,  los hiperparametros que se van a optimizar
# por favor, no desesperarse por lo complejo
obj.fun <- makeSingleObjectiveFunction(
  fn= funcion_optimizar,
  minimize= FALSE, # estoy Maximizando la ganancia
  noisy= TRUE,
  par.set= PARAM$hyperparametertuning$hs,
  has.simple.signature= FALSE
)

ctrl <- makeMBOControl(save.on.disk.at.time= 600, save.file.path= kbayesiana)

ctrl <- setMBOControlTermination(
  ctrl,
  iters= PARAM$hyperparametertuning$iteraciones
)

ctrl <- setMBOControlInfill(ctrl, crit= makeMBOInfillCritEI())

surr.km <- makeLearner(
  "regr.km",
  predict.type= "se",
  covtype= "matern3_2",
  control= list(trace= TRUE)
)


In [80]:
# inicio la optimizacion bayesiana

if (!file.exists(kbayesiana)) {
  run <- mbo(obj.fun, learner= surr.km, control= ctrl)
} else {
  run <- mboContinue(kbayesiana)
} # retomo en caso que ya exista

Computing y column(s) for design. Not provided.



Growing trees.. Progress: 78%. Estimated remaining time: 8 seconds.
Growing trees.. Progress: 80%. Estimated remaining time: 7 seconds.
Growing trees.. Progress: 79%. Estimated remaining time: 8 seconds.
Growing trees.. Progress: 78%. Estimated remaining time: 8 seconds.
Growing trees.. Progress: 81%. Estimated remaining time: 7 seconds.
20260907 110413	148	13	510	33	5	452265000	1
20260907 110413	148	13	510	33	5	452265000	1
Growing trees.. Progress: 63%. Estimated remaining time: 17 seconds.
Growing trees.. Progress: 63%. Estimated remaining time: 18 seconds.
Growing trees.. Progress: 62%. Estimated remaining time: 18 seconds.
Growing trees.. Progress: 62%. Estimated remaining time: 18 seconds.
Growing trees.. Progress: 65%. Estimated remaining time: 16 seconds.
20260907 110836	298	18	491	17	5	460487500	2
20260907 110836	298	18	491	17	5	460487500	2
20260907 111010	338	30	176	3	5	457242500	3
Growing trees.. Progress: 26%. Estimated remaining time: 1 minute, 27 seconds.
Growing trees.. P

[mbo] 0: num.trees=148; max.depth=13; min.node.size=510; mtry=33 : y = 4.52e+08 : 207.8 secs : initdesign

[mbo] 0: num.trees=298; max.depth=18; min.node.size=491; mtry=17 : y = 4.6e+08 : 263.7 secs : initdesign

[mbo] 0: num.trees=338; max.depth=30; min.node.size=176; mtry=3 : y = 4.57e+08 : 94.1 secs : initdesign

[mbo] 0: num.trees=420; max.depth=10; min.node.size=720; mtry=49 : y = 4.48e+08 : 635.9 secs : initdesign

[mbo] 0: num.trees=398; max.depth=1; min.node.size=371; mtry=23 : y = 3.21e+08 : 28.9 secs : initdesign

[mbo] 0: num.trees=39; max.depth=15; min.node.size=121; mtry=41 : y = 4.05e+08 : 83.4 secs : initdesign

[mbo] 0: num.trees=120; max.depth=22; min.node.size=620; mtry=38 : y = 4.42e+08 : 264.7 secs : initdesign

[mbo] 0: num.trees=218; max.depth=17; min.node.size=872; mtry=21 : y = 4.61e+08 : 231.1 secs : initdesign

[mbo] 0: num.trees=107; max.depth=2; min.node.size=886; mtry=15 : y = 3.83e+08 : 16.2 secs : initdesign

[mbo] 0: num.trees=485; max.depth=20; min.node

20260907 120840	497	19	795	2	5	441925000	17


[mbo] 1: num.trees=497; max.depth=19; min.node.size=795; mtry=2 : y = 4.42e+08 : 105.8 secs : infill_ei

Warning message in generateDesign(control$infill.opt.focussearch.points, ps.local, :
“generateDesign could only produce 999 points instead of 1000!”


...

[mbo] 3: num.trees=233; max.depth=12; min.node.size=985; mtry=4 : y = 4.48e+08 : 59.8 secs : infill_ei



20260907 121227	241	20	264	2	5	443767500	20


[mbo] 4: num.trees=241; max.depth=20; min.node.size=264; mtry=2 : y = 4.44e+08 : 59.1 secs : infill_ei



Growing trees.. Progress: 32%. Estimated remaining time: 1 minute, 4 seconds.
Growing trees.. Progress: 64%. Estimated remaining time: 34 seconds.
Growing trees.. Progress: 96%. Estimated remaining time: 3 seconds.
Growing trees.. Progress: 31%. Estimated remaining time: 1 minute, 8 seconds.
Growing trees.. Progress: 63%. Estimated remaining time: 36 seconds.
Growing trees.. Progress: 94%. Estimated remaining time: 5 seconds.
Growing trees.. Progress: 31%. Estimated remaining time: 1 minute, 7 seconds.
Growing trees.. Progress: 62%. Estimated remaining time: 37 seconds.
Growing trees.. Progress: 94%. Estimated remaining time: 5 seconds.
Growing trees.. Progress: 32%. Estimated remaining time: 1 minute, 7 seconds.
Growing trees.. Progress: 63%. Estimated remaining time: 37 seconds.
Growing trees.. Progress: 94%. Estimated remaining time: 5 seconds.
Growing trees.. Progress: 31%. Estimated remaining time: 1 minute, 7 seconds.
Growing trees.. Progress: 63%. Estimated remaining time: 36 se

[mbo] 5: num.trees=500; max.depth=16; min.node.size=592; mtry=21 : y = 4.54e+08 : 517.7 secs : infill_ei

Saved the final state in the file HT450.RDATA

Saved the final state in the file HT450.RDATA



In [81]:
# analizo la salida de la bayesiana

tb_bayesiana <- fread(klog)
setorder( tb_bayesiana, -ganancia)
tb_bayesiana

fecha,num.trees,max.depth,min.node.size,mtry,xval_folds,ganancia,iteracion
<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
20260907 113604,485,20,396,11,5,461615000,10
20260907 113054,218,17,872,21,5,461285000,8
20260907 110836,298,18,491,17,5,460487500,2
20260907 111010,338,30,176,3,5,457242500,3
20260907 122105,500,16,592,21,5,453502500,21
20260907 110413,148,13,510,33,5,452265000,1
20260907 112046,420,10,720,49,5,448497500,4
20260907 120654,183,9,287,6,5,448387500,16
20260907 121027,472,30,658,2,5,447727500,18


In [84]:
# mejores parametros

print( tb_bayesiana[1] )

             fecha num.trees max.depth min.node.size mtry xval_folds  ganancia
1: 20260907 113604       485        20           396   11          5 461615000
   iteracion
1:        10


In [83]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Mon Sep 07 12:21:05 2026"

Una vez que se impriman los mejores hiperparametros proceder:


1.   Copielos en el punto 4.04  en la celda donde se define PARAM
2.   Ejecute el punto 4.04 con estos hiperparámetros optimos
2.   Reporte el resultado en la hoja  **C4-RF Bayesiana**  de la  planilla colaborativa





---

